# Tugas Mata Kuliah Penambangan Data – Reproduksi Analisis Explainability Skforecast

**Nama:** Zakaria Mujur Prasetyo  
**NIM:** 240411100144  
**Mata Kuliah:** Penambangan Data Kelas A  
**Sumber reproduksi:** https://skforecast.org/0.15.1/user_guides/explainability.html

Pada tugas ini, saya mereproduksi studi kasus dari dokumentasi resmi Skforecast tentang **model explainability** pada data *time series*.  
Kasus yang saya angkat adalah prediksi permintaan listrik harian di Victoria, Australia menggunakan `ForecasterRecursive` dan `LGBMRegressor`.  
Saya memilih kasus ini karena relevan dengan topik *penambangan data*, khususnya pada aspek bagaimana model machine learning bisa dijelaskan hasilnya, bukan sekadar menghasilkan angka prediksi.

Pertanyaan tugas yang saya jawab dalam notebook ini:
1. Analisa prediksi tentang apa?
2. Bagaimana bentuk data trainingnya, apa input dan outputnya?
3. Apa itu lag?
4. Jelaskan proses analysis yang dilakukan dari kasus tersebut.

---

## Daftar Isi

**Bagian A – Reproduksi Kode**

1. [Instalasi Library](#1.-Instalasi-Library)
2. [Import Library](#2.-Import-Library)
3. [Mengambil Dataset](#3.-Mengambil-Dataset)
4. [Agregasi Data ke Harian](#4.-Agregasi-Data-ke-Harian)
5. [Visualisasi Data Harian](#5.-Visualisasi-Data-Harian)
6. [Split Data Training dan Testing](#6.-Split-Data-Training-dan-Testing)
7. [Membuat dan Melatih Model Forecasting](#7.-Membuat-dan-Melatih-Model-Forecasting)
8. [Membentuk Data Training Model](#8.-Membentuk-Data-Training-Model)
9. [Model-specific Feature Importance](#9.-Model-specific-Feature-Importance)
10. [SHAP Values](#10.-SHAP-Values)
11. [Explain Individual Observation](#11.-Explain-Individual-Observation)
12. [SHAP Dependence Plot](#12.-SHAP-Dependence-Plot)
13. [Prediksi 10 Hari ke Depan](#13.-Prediksi-10-Hari-ke-Depan)
14. [Membuat Matriks Input untuk Prediksi](#14.-Membuat-Matriks-Input-untuk-Prediksi)
15. [Menjelaskan Nilai Prediksi dengan SHAP](#15.-Menjelaskan-Nilai-Prediksi-dengan-SHAP)
16. [Permutation Feature Importance](#16.-Permutation-Feature-Importance)
17. [Partial Dependence Plot](#17.-Partial-Dependence-Plot)

**Bagian B – Jawaban Pertanyaan Tugas**

- [Pertanyaan 1: Analisa prediksi tentang apa?](#Pertanyaan-1:-Analisa-prediksi-tentang-apa?)
- [Pertanyaan 2: Bentuk data training, input dan output](#Pertanyaan-2:-Bagaimana-bentuk-data-trainingnya,-apa-input-dan-outputnya?)
- [Pertanyaan 3: Apa itu lag?](#Pertanyaan-3:-Apa-itu-lag?)
- [Pertanyaan 4: Proses analysis](#Pertanyaan-4:-Jelaskan-proses-analysis-yang-dilakukan)

**Bagian C – Ringkasan Akhir**

- [Ringkasan Akhir](#Ringkasan-Akhir)

---

## 1. Instalasi Library

Pertama-tama, saya perlu menginstal beberapa library yang dibutuhkan.  
Saya menjalankan notebook ini di Google Colab, jadi saya menginstal library di sini.  
Kalau teman-teman menjalankan di localhost dan library-nya sudah terpasang, bagian ini bisa dilewati saja.

In [ ]:
!pip install -q skforecast==0.15.1 lightgbm shap scikit-learn pandas numpy matplotlib

## 2. Import Library

Setelah instalasi selesai, saya meng-import semua library yang saya butuhkan.  
Library yang saya pakai mengikuti dokumentasi Skforecast, yaitu `pandas` untuk manipulasi data, `matplotlib` untuk visualisasi, `shap` untuk explainability, `sklearn` untuk evaluasi model, `lightgbm` sebagai algoritma machine learning, dan `skforecast` sebagai framework forecasting.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from lightgbm import LGBMRegressor
from sklearn.inspection import permutation_importance
from sklearn.inspection import PartialDependenceDisplay
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

from skforecast.datasets import fetch_dataset
from skforecast.recursive import ForecasterRecursive

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 3. Mengambil Dataset

Di sini saya mengambil dataset `vic_electricity` yang sudah disediakan oleh Skforecast.  
Dataset ini berisi data permintaan listrik di Victoria, Australia. Saya menggunakan kolom-kolom berikut:

| Kolom | Keterangan |
|---|---|
| `Demand` | jumlah permintaan listrik |
| `Temperature` | suhu rata-rata di Melbourne |
| `Date` | tanggal pencatatan |
| `Holiday` | penanda apakah hari tersebut hari libur |

Saya memilih dataset ini karena datanya cukup lengkap dan cocok untuk menunjukkan bagaimana *time series forecasting* bekerja dalam konteks penambangan data.

In [ ]:
data = fetch_dataset(name="vic_electricity")

print("Ukuran data awal:", data.shape)
display(data.head())
display(data.tail())

## 4. Agregasi Data ke Harian

Data aslinya memiliki interval pencatatan setiap setengah jam, jadi terlalu detail kalau langsung dipakai.  
Maka dari itu, saya mengubah data ini menjadi data harian dengan cara:

- `Demand` saya jumlahkan per hari (karena kita ingin tahu total permintaan listrik satu hari penuh).
- `Temperature` saya hitung rata-ratanya per hari (karena kita ingin tahu suhu rata-rata harian).

Ini adalah salah satu tahap *preprocessing* yang penting dalam penambangan data sebelum data dimasukkan ke model.

In [ ]:
data_daily = data.resample("D").agg({
    "Demand": "sum",
    "Temperature": "mean"
})

print("Ukuran data setelah agregasi harian:", data_daily.shape)
display(data_daily.head())
display(data_daily.tail())

## 5. Visualisasi Data Harian

Selanjutnya saya membuat grafik untuk melihat pola permintaan listrik dan suhu dari waktu ke waktu.  
Dari visualisasi ini, saya bisa mengamati apakah ada pola musiman atau tren tertentu yang bisa dimanfaatkan oleh model.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
data_daily["Demand"].plot(ax=ax)
ax.set_title("Permintaan Listrik Harian")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Demand")
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
data_daily["Temperature"].plot(ax=ax)
ax.set_title("Suhu Harian")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Temperature")
plt.show()

## 6. Split Data Training dan Testing

Di tahap ini saya membagi data menjadi dua bagian:

| Bagian | Rentang |
|---|---|
| Data training | sampai 2014-12-21 |
| Data testing | mulai 2014-12-22 |

Data training saya gunakan untuk melatih model, sedangkan data testing saya pakai untuk menguji seberapa akurat prediksi model.  
Pembagian ini penting dalam penambangan data agar kita bisa mengevaluasi performa model secara objektif, yaitu dengan data yang belum pernah dilihat oleh model sebelumnya.

In [ ]:
data_train = data_daily.loc[: "2014-12-21"]
data_test  = data_daily.loc["2014-12-22":]

print("Jumlah data training:", data_train.shape)
print("Jumlah data testing :", data_test.shape)

display(data_train.tail())
display(data_test.head())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
data_train["Demand"].plot(ax=ax, label="Training")
data_test["Demand"].plot(ax=ax, label="Testing")
ax.set_title("Pembagian Data Training dan Testing")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Demand")
ax.legend()
plt.show()

## 7. Membuat dan Melatih Model Forecasting

Di sini saya membuat model peramalan menggunakan:

- `ForecasterRecursive` sebagai metode peramalan multi-step secara rekursif.
- `LGBMRegressor` sebagai algoritma machine learning berbasis pohon keputusan.
- `lags = 7`, artinya saya menggunakan 7 nilai permintaan listrik sebelumnya sebagai fitur.
- `Temperature` saya tambahkan sebagai variabel eksogen (variabel tambahan dari luar target).

Target yang saya prediksi adalah `Demand`, yaitu permintaan listrik harian.  
Saya memilih LightGBM karena algoritma ini dikenal cepat dan cocok untuk data tabular seperti kasus ini.

In [ ]:
forecaster = ForecasterRecursive(
    regressor = LGBMRegressor(random_state=123, verbose=-1),
    lags      = 7
)

forecaster.fit(
    y    = data_train["Demand"],
    exog = data_train["Temperature"]
)

forecaster

## 8. Membentuk Data Training Model

Pada bagian ini, saya ingin menunjukkan bagaimana Skforecast mengubah data *time series* menjadi bentuk tabel *supervised learning*.  
Ini adalah konsep penting dalam penambangan data, yaitu **transformasi data** agar bisa diolah oleh algoritma machine learning.

Input model yang terbentuk terdiri dari:

- `lag_1` sampai `lag_7` (nilai Demand hari-hari sebelumnya)
- `Temperature` (suhu pada hari yang diprediksi)

Sedangkan output atau target modelnya adalah:

- `y`, yaitu nilai `Demand` yang ingin saya prediksi.

In [ ]:
X_train, y_train = forecaster.create_train_X_y(
    y    = data_train["Demand"],
    exog = data_train["Temperature"]
)

print("Bentuk X_train:", X_train.shape)
print("Bentuk y_train:", y_train.shape)

display(X_train.head())
display(y_train.head())

### Penjelasan Bentuk Data Training

Untuk lebih jelasnya, berikut arti dari setiap kolom yang saya dapatkan:

| Kolom | Arti |
|---|---|
| `lag_1` | Demand 1 hari sebelumnya |
| `lag_2` | Demand 2 hari sebelumnya |
| `lag_3` | Demand 3 hari sebelumnya |
| `lag_4` | Demand 4 hari sebelumnya |
| `lag_5` | Demand 5 hari sebelumnya |
| `lag_6` | Demand 6 hari sebelumnya |
| `lag_7` | Demand 7 hari sebelumnya |
| `Temperature` | suhu pada hari yang diprediksi |
| `y` | Demand aktual yang menjadi target prediksi |

Jadi bisa saya simpulkan bahwa model ini belajar dari pola 7 hari terakhir ditambah informasi suhu untuk memprediksi permintaan listrik hari berikutnya.

## 9. Model-specific Feature Importance

Di sini saya melihat fitur mana yang dianggap paling penting oleh model `LGBMRegressor`.  
Feature importance ini merupakan salah satu teknik *explainability* dalam penambangan data yang membantu saya memahami alasan di balik prediksi model.

In [ ]:
feature_importance = forecaster.get_feature_importances()
display(feature_importance)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
feature_importance.sort_values("importance").plot(
    x="feature",
    y="importance",
    kind="barh",
    ax=ax,
    legend=False
)
ax.set_title("Feature Importance dari LGBMRegressor")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")
plt.show()

## 10. SHAP Values

Selanjutnya saya menggunakan SHAP (SHapley Additive exPlanations) untuk menjelaskan kontribusi setiap fitur terhadap hasil prediksi model.  
SHAP ini berbasis teori permainan dan bisa menunjukkan seberapa besar pengaruh masing-masing fitur, baik positif maupun negatif.  
Karena model saya berbasis pohon keputusan (LightGBM), saya menggunakan `TreeExplainer` yang lebih cepat.

In [ ]:
shap.initjs()

explainer = shap.TreeExplainer(forecaster.regressor)
shap_values_train = explainer.shap_values(X_train)

# Untuk menjaga kompatibilitas jika output SHAP berbentuk list
if isinstance(shap_values_train, list):
    shap_values_train = shap_values_train[0]

print("Bentuk SHAP values:", shap_values_train.shape)

### SHAP Summary Plot dalam Bentuk Bar

Plot bar ini menunjukkan rata-rata pengaruh setiap fitur terhadap prediksi.  
Dari sini saya bisa melihat fitur mana yang secara umum paling berpengaruh terhadap model.

In [ ]:
shap.summary_plot(shap_values_train, X_train, plot_type="bar")

### SHAP Summary Plot Detail

Berbeda dengan plot bar sebelumnya, plot ini menunjukkan **arah pengaruh** fitur.  
Saya bisa melihat apakah nilai fitur yang tinggi mendorong prediksi naik atau turun.  
Warna merah menunjukkan nilai fitur tinggi, sedangkan biru menunjukkan nilai rendah.

In [ ]:
shap.summary_plot(shap_values_train, X_train)

## 11. Explain Individual Observation

Di bagian ini saya mencoba menjelaskan satu data training secara individual.  
Tujuannya agar saya bisa melihat fitur mana yang menaikkan atau menurunkan hasil prediksi pada satu baris data tertentu.  
Ini yang disebut sebagai **local explanation** dalam konteks explainability.

In [ ]:
shap.force_plot(
    explainer.expected_value,
    shap_values_train[0, :],
    X_train.iloc[0, :]
)

## 12. SHAP Dependence Plot

Dependence plot ini saya gunakan untuk melihat hubungan antara satu fitur dengan nilai SHAP-nya.  
Pada contoh ini saya memilih fitur `Temperature` karena saya ingin tahu bagaimana suhu memengaruhi prediksi permintaan listrik.  
Hasilnya cukup menarik karena bisa terlihat pola non-linear.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
shap.dependence_plot("Temperature", shap_values_train, X_train, ax=ax)
plt.show()

## 13. Prediksi 10 Hari ke Depan

Sekarang saya menggunakan model yang sudah dilatih untuk memprediksi 10 hari pertama pada data testing.  
Saya ingin melihat seberapa akurat model dalam memprediksi permintaan listrik di hari-hari yang belum pernah dilihat sebelumnya.

In [ ]:
predictions = forecaster.predict(
    steps = 10,
    exog  = data_test["Temperature"]
)

display(predictions)

In [ ]:
hasil_prediksi = pd.DataFrame({
    "Actual_Demand": data_test["Demand"].iloc[:10],
    "Predicted_Demand": predictions
})

hasil_prediksi["Error"] = hasil_prediksi["Actual_Demand"] - hasil_prediksi["Predicted_Demand"]

display(hasil_prediksi)

In [ ]:
mae = mean_absolute_error(hasil_prediksi["Actual_Demand"], hasil_prediksi["Predicted_Demand"])
rmse = np.sqrt(mean_squared_error(hasil_prediksi["Actual_Demand"], hasil_prediksi["Predicted_Demand"]))
mape = mean_absolute_percentage_error(hasil_prediksi["Actual_Demand"], hasil_prediksi["Predicted_Demand"]) * 100

print("Evaluasi prediksi 10 hari pertama data testing")
print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"MAPE : {mape:.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
hasil_prediksi[["Actual_Demand", "Predicted_Demand"]].plot(ax=ax)
ax.set_title("Perbandingan Demand Aktual dan Prediksi")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Demand")
plt.show()

## 14. Membuat Matriks Input untuk Prediksi

Di sini saya membuat matriks `X_predict` yang berisi fitur-fitur yang dipakai model ketika melakukan prediksi.  
Kolomnya sama seperti data training, yaitu `lag_1` sampai `lag_7` dan `Temperature`.  
Ini berguna untuk saya analisis lebih lanjut menggunakan SHAP di langkah berikutnya.

In [ ]:
X_predict = forecaster.create_predict_X(
    steps = 10,
    exog  = data_test["Temperature"]
)

display(X_predict)

## 15. Menjelaskan Nilai Prediksi dengan SHAP

Pada bagian ini saya menjelaskan prediksi pada tanggal tertentu, yaitu `2014-12-22`, menggunakan SHAP.  
Dengan cara ini saya bisa memahami mengapa model memberikan prediksi tertentu pada tanggal tersebut — fitur mana yang mendorong nilai naik dan mana yang menariknya turun.

In [ ]:
predicted_date = "2014-12-22"
iloc_predicted_date = X_predict.index.get_loc(predicted_date)

shap_values_predict = explainer.shap_values(X_predict)
if isinstance(shap_values_predict, list):
    shap_values_predict = shap_values_predict[0]

shap.force_plot(
    explainer.expected_value,
    shap_values_predict[iloc_predicted_date, :],
    X_predict.iloc[iloc_predicted_date, :]
)

## 16. Permutation Feature Importance

Selain SHAP, saya juga menggunakan metode *permutation importance*.  
Cara kerjanya adalah dengan mengacak nilai satu fitur, lalu melihat seberapa besar performa model menurun.  
Kalau performa model turun banyak setelah fitur diacak, berarti fitur itu memang penting buat model.  
Metode ini saya anggap lebih "jujur" karena model-agnostic, artinya bisa dipakai untuk model apa saja.

In [ ]:
r = permutation_importance(
    estimator    = forecaster.regressor,
    X            = X_train,
    y            = y_train,
    n_repeats    = 3,
    max_samples  = 0.5,
    random_state = 123
)

permutation_importances = pd.DataFrame({
    "feature": X_train.columns,
    "mean_importance": r.importances_mean,
    "std_importance": r.importances_std
}).sort_values("mean_importance", ascending=False)

display(permutation_importances)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
permutation_importances.sort_values("mean_importance").plot(
    x="feature",
    y="mean_importance",
    kind="barh",
    ax=ax,
    legend=False
)
ax.set_title("Permutation Feature Importance")
ax.set_xlabel("Mean Importance")
ax.set_ylabel("Feature")
plt.show()

## 17. Partial Dependence Plot

Terakhir, saya membuat Partial Dependence Plot (PDP) untuk melihat hubungan antara fitur tertentu dengan hasil prediksi model.  
Saya memilih fitur `Temperature` dan `lag_1` karena dari analisis sebelumnya kedua fitur ini termasuk yang paling berpengaruh.  
PDP ini membantu saya memahami bagaimana perubahan nilai suatu fitur memengaruhi prediksi secara rata-rata.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

PartialDependenceDisplay.from_estimator(
    estimator = forecaster.regressor,
    X         = X_train,
    features  = ["Temperature", "lag_1"],
    kind      = "both",
    ax        = ax
)

ax.set_title("Partial Dependence Plot")
fig.tight_layout()
plt.show()

---

# Jawaban Pertanyaan Tugas

Berikut adalah jawaban saya terhadap pertanyaan-pertanyaan yang diberikan dalam tugas mata kuliah **Penambangan Data Kelas A**.

## Pertanyaan 1: Analisa prediksi tentang apa?

Menurut pemahaman saya, analisis prediksi pada kasus ini membahas tentang **prediksi permintaan (demand) listrik harian** di wilayah Victoria, Australia.

Secara lebih rinci:

- **Variabel yang diprediksi (target):** `Demand`, yaitu total kebutuhan atau permintaan listrik dalam satu hari.
- **Tujuan prediksi:** Memperkirakan berapa besar permintaan listrik di hari-hari mendatang berdasarkan pola permintaan di masa lalu dan kondisi suhu.
- **Metode yang digunakan:** Model `ForecasterRecursive` dari library Skforecast dengan algoritma `LGBMRegressor` (LightGBM).
- **Jenis analisis:** Selain prediksi, kasus ini juga fokus pada **explainability** (kemampuan menjelaskan model), yaitu memahami *mengapa* model memberikan prediksi tertentu.

Dalam konteks mata kuliah Penambangan Data, kasus ini relevan karena menunjukkan bagaimana teknik *data mining* dapat diaplikasikan pada data *time series* untuk menghasilkan prediksi yang **bukan hanya akurat, tetapi juga bisa dijelaskan**. Ini penting karena dalam dunia nyata, kita tidak cukup hanya mengetahui "angka prediksinya berapa", tetapi juga perlu tahu "kenapa model memberikan angka itu".

## Pertanyaan 2: Bagaimana bentuk data trainingnya, apa input dan outputnya?

Data awal yang saya gunakan berbentuk *time series* (data runtun waktu), yaitu data yang tersusun berdasarkan urutan waktu.  
Setelah saya ubah dari interval setengah jam menjadi data harian, library Skforecast secara otomatis mengubah data ini menjadi format **supervised learning** — yaitu format tabel yang terdiri dari kolom input (fitur) dan kolom output (target).

### Input Model (Fitur)

| Fitur | Keterangan | Tipe |
|---|---|---|
| `lag_1` | Demand 1 hari sebelumnya | Lag feature |
| `lag_2` | Demand 2 hari sebelumnya | Lag feature |
| `lag_3` | Demand 3 hari sebelumnya | Lag feature |
| `lag_4` | Demand 4 hari sebelumnya | Lag feature |
| `lag_5` | Demand 5 hari sebelumnya | Lag feature |
| `lag_6` | Demand 6 hari sebelumnya | Lag feature |
| `lag_7` | Demand 7 hari sebelumnya | Lag feature |
| `Temperature` | Suhu rata-rata harian di Melbourne | Variabel eksogen |

### Output Model (Target)

| Output | Keterangan |
|---|---|
| `y` | Nilai Demand aktual yang ingin diprediksi |

### Ilustrasi

Misalnya, untuk memprediksi permintaan listrik pada tanggal **8 Januari 2012**, model akan menggunakan:
- Permintaan listrik tanggal 7 Januari (`lag_1`), 6 Januari (`lag_2`), ..., hingga 1 Januari (`lag_7`)
- Suhu rata-rata pada tanggal 8 Januari (`Temperature`)

Jadi, bentuk data training saya adalah **tabel** dengan 8 kolom input (`lag_1` s.d. `lag_7` + `Temperature`) dan 1 kolom output (`y`).  
Proses transformasi dari time series ke tabel ini merupakan contoh nyata dari **feature engineering** dalam penambangan data.

## Pertanyaan 3: Apa itu lag?

Dari yang saya pelajari, **lag** (atau *lagged variable*) adalah nilai data pada **periode sebelumnya** yang digunakan sebagai input untuk memprediksi nilai saat ini atau masa depan.

### Penjelasan Sederhana

Bayangkan kita ingin memprediksi permintaan listrik **hari ini**. Salah satu informasi yang berguna adalah mengetahui permintaan listrik **kemarin**, **2 hari lalu**, **3 hari lalu**, dan seterusnya. Nah, nilai-nilai di masa lalu inilah yang disebut "lag".

### Dalam Kasus Ini

| Lag | Arti |
|---|---|
| `lag_1` | Demand **1 hari** sebelumnya |
| `lag_2` | Demand **2 hari** sebelumnya |
| `lag_3` | Demand **3 hari** sebelumnya |
| `lag_4` | Demand **4 hari** sebelumnya |
| `lag_5` | Demand **5 hari** sebelumnya |
| `lag_6` | Demand **6 hari** sebelumnya |
| `lag_7` | Demand **7 hari** sebelumnya |

Karena saya memakai `lags = 7`, artinya model saya menggunakan data permintaan listrik selama **7 hari terakhir** sebagai "jendela" (*window*) untuk memprediksi permintaan listrik hari berikutnya.

### Mengapa Lag Penting?

- **Menangkap pola temporal:** Lag membantu model mengenali pola berulang, misalnya permintaan listrik yang selalu tinggi di hari kerja dan rendah di akhir pekan.
- **Mengubah time series jadi tabel:** Dengan lag, data time series bisa diubah menjadi format supervised learning yang bisa diolah oleh algoritma machine learning biasa.
- **Konsep fundamental:** Dalam penambangan data dan forecasting, lag adalah salah satu teknik paling dasar dan paling sering digunakan untuk menangkap ketergantungan antar waktu (*temporal dependency*).

## Pertanyaan 4: Jelaskan proses analysis yang dilakukan

Berikut adalah proses analisis yang saya lakukan secara bertahap:

### Tahap 1: Pengambilan Data
Saya mengambil dataset `vic_electricity` dari library Skforecast. Dataset ini berisi data permintaan listrik, suhu, tanggal, dan informasi hari libur di Victoria, Australia. Data awalnya berjumlah sangat banyak karena dicatat setiap setengah jam.

### Tahap 2: Preprocessing Data
Saya melakukan *preprocessing* dengan mengagregasi data menjadi data harian:
- Kolom `Demand` dijumlahkan per hari (untuk mendapat total permintaan satu hari penuh)
- Kolom `Temperature` dihitung rata-ratanya per hari

Tahap ini penting karena data yang terlalu granular bisa menyulitkan model dan membuat proses training lebih lama.

### Tahap 3: Eksplorasi & Visualisasi Data
Saya membuat grafik time series untuk melihat pola dari data `Demand` dan `Temperature`. Dari visualisasi ini, saya bisa mengamati adanya **pola musiman** (seasonal pattern) di mana permintaan listrik naik di musim panas dan dingin.

### Tahap 4: Split Data Training & Testing
Data saya bagi menjadi:
- **Training:** sampai 2014-12-21 (untuk melatih model)
- **Testing:** mulai 2014-12-22 (untuk menguji prediksi model)

Pembagian ini sesuai dengan prinsip evaluasi objektif dalam penambangan data, di mana model diuji pada data yang belum pernah dilihat sebelumnya.

### Tahap 5: Pembuatan & Training Model
Saya membuat model `ForecasterRecursive` dengan algoritma `LGBMRegressor`, menggunakan 7 lag dan variabel eksogen `Temperature`. Model dilatih menggunakan data training.

### Tahap 6: Transformasi Data ke Supervised Learning
Skforecast secara otomatis mengubah time series menjadi tabel supervised learning dengan kolom `lag_1` s.d. `lag_7` + `Temperature` sebagai input dan `Demand` sebagai output.

### Tahap 7: Analisis Explainability
Ini adalah **bagian inti** dari kasus ini. Saya menggunakan beberapa metode untuk menjelaskan model:

| Metode | Penjelasan |
|---|---|
| **Feature Importance** | Menunjukkan fitur mana yang paling sering dipakai oleh model LightGBM dalam proses splitting pohon keputusan |
| **SHAP Values** | Menunjukkan kontribusi setiap fitur secara detail. SHAP Summary Plot (bar) untuk gambaran umum, SHAP Summary Plot (dot) untuk arah pengaruh, dan Force Plot untuk penjelasan per observasi |
| **SHAP Dependence Plot** | Menunjukkan hubungan non-linear antara satu fitur (misalnya `Temperature`) dengan pengaruhnya terhadap prediksi |
| **Permutation Importance** | Mengukur pentingnya fitur dengan cara mengacak nilainya dan melihat seberapa besar performa model menurun. Metode ini model-agnostic |
| **Partial Dependence Plot (PDP)** | Menunjukkan bagaimana perubahan nilai satu fitur memengaruhi prediksi secara rata-rata, dengan mempertahankan fitur lain konstan |

### Tahap 8: Prediksi & Evaluasi
Model saya gunakan untuk memprediksi 10 hari pertama pada data testing. Hasilnya saya bandingkan dengan data aktual dan evaluasi menggunakan metrik:
- **MAE (Mean Absolute Error):** rata-rata kesalahan absolut
- **RMSE (Root Mean Squared Error):** akar dari rata-rata kuadrat kesalahan
- **MAPE (Mean Absolute Percentage Error):** rata-rata kesalahan dalam persentase

### Kesimpulan Proses
Keseluruhan proses ini menunjukkan bahwa model machine learning tidak hanya bisa dipakai untuk forecasting, tetapi juga bisa **dijelaskan hasilnya** menggunakan teknik explainability. Ini sejalan dengan apa yang saya pelajari di mata kuliah Penambangan Data, bahwa **memahami model sama pentingnya dengan membangun model itu sendiri**.

---

# Ringkasan Akhir

Dari reproduksi yang saya lakukan dalam tugas Penambangan Data ini, saya dapat menyimpulkan bahwa:

1. **Target prediksi** dalam kasus ini adalah `Demand` (permintaan listrik harian di Victoria, Australia).
2. **Input model** terdiri dari `lag_1` sampai `lag_7` (nilai permintaan listrik 7 hari terakhir) dan `Temperature` (suhu harian rata-rata). Total ada **8 fitur input**.
3. **Output model** adalah prediksi `Demand` untuk hari berikutnya.
4. **Lag** adalah nilai masa lalu yang digunakan sebagai fitur input. Konsep ini sangat fundamental dalam analisis time series karena membantu model mengenali pola temporal.
5. **Analisis explainability** yang saya lakukan (feature importance, SHAP, permutation importance, dan PDP) menunjukkan bahwa `lag_1` dan `Temperature` adalah fitur yang paling berpengaruh terhadap prediksi model.

Tugas ini memberikan saya pemahaman yang lebih mendalam tentang bagaimana teknik penambangan data dapat diterapkan pada permasalahan nyata, khususnya dalam hal **forecasting** dan **model interpretability**.

---
*Notebook ini dibuat oleh Zakaria Mujur Prasetyo (240411100144) untuk tugas mata kuliah Penambangan Data Kelas A.*